# Rebuttal experiments

## Uncertainty quantification baselines: AUROCS of Semantic Entropy and Kernel Language Entropy on TriviaQA

In [ ]:
from functions import *
import os
if not os.path.exists('figures'):
    os.makedirs('figures')
if not os.path.exists('results'):
    os.makedirs('results')

In [ ]:
n_bootstrap = 20
datasets = ['TriviaQA_2k']
models = ['phi_4', 'phi_4_mini', 'llama4_maverick']
uncertainty_methods = ['kle', 'se']

In [ ]:
results = auroc_experiments_other_uncertainties(
    datasets=datasets,
    models=models,
    uncertainty_methods=uncertainty_methods,
    correctness_label='fuzzy_correctness',
    n_bootstrap=n_bootstrap
)
results = results.drop('seed', axis=1)
results['dataset'] = results['dataset'].replace({'TriviaQA_2k': 'TriviaQA', 'OpenNQ_2k': 'Natural Questions'})
results['model'] = results['model'].replace({'phi_4_mini': 'Phi 4 Mini', 'phi_4': 'Phi 4', 'llama4_maverick': 'Llama4 Maverick'})
results

In [ ]:
results.drop('dataset', axis=1, inplace=True)
table_means = results.groupby(['model', 'uncertainty_method']).mean().round(3).astype(str)
table_stds = results.groupby(['model', 'uncertainty_method']).std().div(n_bootstrap).round(3).astype(str)
table_pm = table_means + ' $\\pm$ ' + table_stds
#table_pm = table_pm.reset_index().pivot_table(index='model', columns='uncertainty_method', values='auroc')
print(table_pm.to_latex())
table_pm

In [ ]:
# Table 2
table_pivot = table_pm.reset_index().pivot(
    index='model',
    columns='uncertainty_method',
    values='auroc'
)
print(table_pivot.to_latex())

table_pivot


## Calibration baseline: Optimizing sampling temperature for correctness calibration using Semantic Entropy confidences

Calibration baseline presented in: Lamb, Tom A., et al. "Semantic-level confidence calibration of language models via temperature scaling." ICLR Workshop: Quantify Uncertainty and Hallucination in Foundation Models: The Next Frontier in Reliable AI. 2025.

In [ ]:
import os
import re
import matplotlib.pyplot as plt

# We do a linear search over 20 sampling temperatures (logspace 0.1 → 3.0).
# phi_4_mini uses all 20 (indices 0-19); llama4_maverick uses the first 17 (indices 0-16).
ALL_TEMPS = np.logspace(np.log10(0.1), np.log10(3.0), num=20)

In [ ]:
datasets = ['TriviaQA']
models = ['phi_4_mini', 'llama4_maverick']
uncertainty_methods = ['se']

optimal_temps = plot_ece_vs_temperature(
    datasets=datasets,
    models=models,
    uncertainty_methods=uncertainty_methods,
    correctness_label='fuzzy_correctness',
    n_bins=15,
)

### Eigenvalue-based ECE and reliability for Phi 4 mini at the optimal temperature t12

In [ ]:
# Figure 10
confs, max_EVs, freq, ece_results = compute_rel_diag('TriviaQA', 'phi_4_mini', None, bin_type='bin2cluster', num_bins=8, num_clusters=5, baseline=True, baseline_temperature_index=12)
plot_single_rel_diag(confs, max_EVs, freq, ece_results, figsize=(2.5, 2), font_size=10, title=None, file_name='rel_diagram_sampling_t_baseline_phi_4_mini_triviaqa.png')

### Eigenvalue-based ECE and reliability for Llama 4 maverick at the optimal temperature t3

In [ ]:
confs, max_EVs, freq, ece_results = compute_rel_diag('TriviaQA', 'llama4_maverick', None, bin_type='bin2cluster', num_bins=8, num_clusters=5, baseline=True, baseline_temperature_index=3)
plot_single_rel_diag(confs, max_EVs, freq, ece_results, figsize=(2.5, 2), font_size=10, title=None)

## Main results using another embedder: jina_v5_nano

In [ ]:
rerun = True
temps = np.round([10**(i/10) for i in range(-1, 10)], 2) #test
embedding_models = ['jina_v5_nano']
models = ['phi_4_mini', 'phi_4', 'llama4_maverick']
datasets = ['TriviaQA', 'OpenNQ']
results=dict()
for embedding_model in embedding_models:
    if rerun:
        results[embedding_model] = compute_TS_curves(datasets, models, temps, embedding_model = embedding_model, save_results=f'results/TS_results_{embedding_model}.json')
    else:
        results[embedding_model] = pd.read_json(f"results/TS_results_{embedding_model}.json")

In [ ]:
TS_dicts = {}
for embedding_model in embedding_models:
    df = pd.DataFrame(results[embedding_model])
    TS_dict_em = {}
    for _, row in df.iterrows():
        dataset = row['dataset']
        model = row['model']
        optimal_temp = np.array(row['temps'])[np.argmin(row['risks'])]
        TS_dict_em.setdefault(dataset, {})[model] = optimal_temp
    TS_dicts[embedding_model] = TS_dict_em

TS_dicts

In [ ]:
embedding_model = 'jina_v5_nano'
# Plot before
# Figure 11 (a)
plot_all_rel_diag(
    datasets, models, bin_type='bin2cluster', 
    num_bins=8, num_clusters=5, embedding_model=embedding_model, file_name=f"rel_diag_{embedding_model}_before.png"
)

In [ ]:
# Plot after
# Figure 11 (b)
plot_all_rel_diag(
    datasets, models, bin_type='bin2cluster', TS_dict=TS_dicts[embedding_model],
    num_bins=8, num_clusters=5, embedding_model=embedding_model, file_name=f"rel_diag_{embedding_model}_after.png"
)

In [ ]:
rerun= True
n_bootstrap = 20
results_dict = dict()
for embedding_model in embedding_models:
    if rerun:
        results_dict[embedding_model] = auroc_experiments(datasets, models, TS_dicts[embedding_model], n_bootstrap=n_bootstrap, embedding_model=embedding_model)
        results_dict[embedding_model].to_pickle(f'results/auroc_results_{embedding_model}.pkl')
    else:
        results_dict[embedding_model] = pd.read_pickle(f'results/auroc_results_{embedding_model}.pkl')

In [ ]:
embedding_model = 'jina_v5_nano'
results = results_dict[embedding_model]
results = results.drop('seed', axis=1)
results = results.rename(columns={
    'ev_auroc': 'Eigenvalue', 'TS_ev_auroc': 'Eigenvalue TS', 'ent_auroc': 'Entropy', 'TS_ent_auroc': 'Entropy TS'
})
results['dataset'] = results['dataset'].replace({'TriviaQA_2k': 'TriviaQA', 'OpenNQ_2k': 'Natural Questions'})
results['model'] = results['model'].replace({'phi_4_mini': 'Phi 4 Mini', 'phi_4': 'Phi 4', 'llama4_maverick': 'Llama4 Maverick'})
results

In [ ]:
# Table 5
table_means = results.groupby(['dataset', 'model']).mean().round(3).astype(str)
table_stds = results.groupby(['dataset', 'model']).std().div(n_bootstrap).round(3).astype(str)
table_pm = table_means + ' $\\pm$ ' + table_stds
print(table_pm.to_latex())
table_pm

## Cosine similarity between d(X) pairs within and outside of clusters

In [ ]:
datasets = ['TriviaQA', 'OpenNQ']
models = ['phi_4_mini', 'phi_4', 'llama4_maverick']
similarities_df = compute_average_pairwise_dX_similarities(datasets, models)

In [ ]:
# Table 4
similarities_grouped = similarities_df.groupby(['dataset', 'model']).mean().round(3).astype(str)
print(similarities_grouped.to_latex())
similarities_grouped


## Risk, Eigenvalue ECE, Estimated matrix calibration error all are co-minimized at the same temperature

In [ ]:
rerun = True
if rerun:
        datasets = ['TriviaQA', 'OpenNQ']
        models = ['phi_4_mini', 'phi_4', 'llama4_maverick']
        temps = np.round([10**(i/10) for i in range(-1, 10)], 2)
        results_df = compute_TS_curves_risks_and_eces(datasets, models, temps, 
                                                        embedding_model='all_mpnet_base_v2', save_results=f'results/risk_vs_ev-ece_vs_matrix-calibration-errors.json')
else:
    results_df = pd.read_json(f'results/risk_vs_ev-ece_vs_matrix-calibration-errors.json', orient='records')
    

In [ ]:
results_df

In [ ]:
# Figure 9
plot_risk_ece_curves(results_df, file_name='risk_vs_ev-ece_vs_matrix-calibration-errors.png')